In [ ]:
from concurrent.futures import ThreadPoolExecutor
import threading
# from classes.Segment import Segment  # 确保导入路径正确
from openai import OpenAI
import json
from openai import OpenAIError
import time

# 初始化OpenAI客户端
openai_model = OpenAI(api_key="", base_url="http://localhost:8765/v1")

# 线程局部存储，确保每个线程有自己的OpenAI客户端实例
thread_local = threading.local()

def get_openai_client():
    if not hasattr(thread_local, 'openai_client'):
        thread_local.openai_client = openai_model
    return thread_local.openai_client

def get_response():
    client = get_openai_client()
    try:
        response = client.chat.completions.create(
            model="claude-3.5-sonnet",
            messages=[{"role": "user", "content": "请直接返回以下格式的纯JSON响应，不要使用任何Markdown格式或代码块:\n{\"status\": 200 表示连接成功，400 表示失败}"}]
        )
        return json.loads(response.choices[0].message.content)
    except json.JSONDecodeError:
        return {"status": 500, "error": "Invalid JSON response"}
    except OpenAIError as e:
        return {"status": 502, "error": str(e)}
    except Exception as e:
        return {"status": 500, "error": str(e)}

def process_segment(segment):
    result = get_response()
    time.sleep(0.1)  # 添加延迟以避免速率限制
    return result

def batch_process_segments(segments, max_workers=5):
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        results = list(executor.map(process_segment, segments))
    return results

print(batch_process_segments([1, 2]))  # 测试批量处理

[{'status': 200}, {'status': 200}]
